In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from pyspark.sql.functions import *
import os
import sys

In [0]:
curr_dir=os.getcwd()
print(curr_dir)

In [0]:

sys.path.append(curr_dir)

In [0]:
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/first_project_catalogy/source_schema_anshu/sorce_data_volume_anshu/customers/")
    

In [0]:
display(df)

In [0]:
schema_customer=df.schema

In [0]:
schema_customer

## spark streaming ### 
#### first read to get the schema cause streaming mode doesn't work with the infere schema 
#### second readStream to open the tap
#### third write is to define the destinations for the tap water

In [0]:
entities=['customers','payments','drivers','locations','trips','vechicles']


In [0]:
for i in entities:
    df_batch=spark.read.format('csv')\
        .option('header',True)\
        .option('inferSchema',True)\
        .load(f"/Volumes/first_project_catalogy/source_schema_anshu/sorce_data_volume_anshu/{i}/")
    schema_entity=df_batch.schema

    df= spark.readStream.format("csv")\
    .option("header","true")\
    .schema(schema_entity)\
    .load(f"/Volumes/first_project_catalogy/source_schema_anshu/sorce_data_volume_anshu/{i}/")
    df.writeStream.format("delta")\
    .option("checkpointLocation",f"/Volumes/first_project_catalogy/bronze_schema/check_point_volume/{i}")\
    .trigger(once=True)\
    .outputMode("append")\
    .toTable(f"first_project_catalogy.bronze_schema.{i}")

### Silver Transformations Satrts From here 

#### Transforming CustomerTables

In [0]:
df_cus = spark.read.table("first_project_catalogy.bronze_schema.customers")

In [0]:
from pyspark.sql.functions import split

In [0]:

df_cus=df_cus.withColumn("domain",split('email','@').getItem(1))
display(df_cus)

In [0]:
import pyspark.sql.functions as F

In [0]:
df_cus=df_cus.withColumn("phone_nnumber",F.regexp_replace('phone_number',r"^[0-9]",""))

In [0]:
display(df_cus)

In [0]:
df_cus=df_cus.drop("phone_nnumber")

In [0]:
df_cus=(df_cus.withColumn("phone", F.regexp_replace(("phone_number"), "[^0-9]", "")))


In [0]:
display(df_cus.limit(5))

In [0]:
display(df_cus.withColumn("full_name",concat(col("first_name"),lit(" "),col("last_name"))))

In [0]:
df_cus=df_cus.withColumn("full_name",concat(col('first_name'),lit(' '),col('last_name')))

In [0]:
display(df_cus)

In [0]:
df_cus=df_cus.drop(col("phone_number"))
display(df_cus.limit(5))

In [0]:
from utills.custom_module import * 


In [0]:
from utills.custom_module import transforms

In [0]:
cust_obj = transforms(spark)
cust_obj_trans=cust_obj.dedup(df_cus,['customer_id'],'last_updated_timestamp')
display(cust_obj_trans.limit(15))

In [0]:
from pyspark.sql import functions as sf
df_cust3=cust_obj_trans.sort(sf.asc('customer_id'))

In [0]:
display(df_cust3.limit(15))

In [0]:
df_cust3=cust_obj.process_time_stamp(df_cust3)

In [0]:
display(df_cust3.limit(15))


In [0]:



if spark.catalog.tableExists("first_project.silver_schema.customer"):
    df.write.formt("delta")\
        .mode("append")\
        .saveAsTable("first_project.silver_schema.customer")
else:
    cust_obj.upsert(df_cust3,['customer_id'],'customers','last_updated_timestamp')


In [0]:
%sql
SELECT COUNT(*) FROM first_project_catalogy.silver_schema.customers

### Drivers

In [0]:
df_driver= spark.read.table("first_project_catalogy.bronze_schema.drivers")

In [0]:
display(df_driver)